##**Step 2: Data Collection and Understanding**

This section documents the dataset source, dataset scope, feature types, data quality checks, target distribution, initial outlier review, and data dictionary.


**a. Dataset Source and Dataset Strategy**


The dataset used in this capstone is the Bank Account Fraud Dataset Suite - NeurIPS 2022, publicly available through Kaggle. The dataset suite contains six synthetic bank account fraud tabular datasets and is designed to evaluate machine learning and fair machine learning methods in fraud detection.


For this capstone, Base.csv will be used as the primary dataset for data understanding, EDA, preprocessing, feature engineering, model development, model comparison, explainability, and business KPI analysis. Variant II.csv will be used as a secondary dataset for robustness and fairness stress-testing. The two datasets will be analyzed separately and will not be merged.

**b. Dataset Loading**

The following code loads both Base.csv and Variant II.csv.

In [ ]:
import sys
import platform
import random
import numpy as np
import pandas as pd

import kagglehub
import os
from kagglehub import KaggleDatasetAdapter
from pathlib import Path

TARGET_COL = "fraud_bool"

# Adjust paths depending on your folder structure
DATA_DIR = Path("data/raw")
os.makedirs(DATA_DIR, exist_ok=True)

DATASET_HANDLE = "sgpjesus/bank-account-fraud-dataset-neurips-2022"

# Load Base.csv
base_file_name = "Base.csv"
base_df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    DATASET_HANDLE,
    base_file_name,
    pandas_kwargs={
        "encoding": "latin1"
    }
)
print(f"Loaded {base_file_name}")

# Load Variant II.csv
variant_ii_file_name = "Variant II.csv"
variant_ii_df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    DATASET_HANDLE,
    variant_ii_file_name,
    pandas_kwargs={
        "encoding": "latin1"
    }
)
print(f"Loaded {variant_ii_file_name}")

# Remove common unnamed index column if present
for data in [base_df, variant_ii_df]:
    if "Unnamed: 0" in data.columns:
        data.drop(columns=["Unnamed: 0"], inplace=True)

print("Base shape:", base_df.shape)
print("Variant II shape:", variant_ii_df.shape)

display(base_df.head())
display(variant_ii_df.head())

Using Colab cache for faster access to the 'bank-account-fraud-dataset-neurips-2022' dataset.
Loaded Base.csv
Using Colab cache for faster access to the 'bank-account-fraud-dataset-neurips-2022' dataset.
Loaded Variant II.csv
Base shape: (1000000, 32)
Variant II shape: (1000000, 32)


,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,...,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,...,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,...,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0
2,0,0.8,0.996707,9,14,40,0.012316,-1.490386,AB,1095,...,0,200.0,0,INTERNET,22.730559,windows,0,1,0,0
3,0,0.6,0.475100,11,14,30,0.006991,-1.863101,AB,3483,...,0,200.0,0,INTERNET,15.215816,linux,1,1,0,0
4,0,0.9,0.842307,-1,29,40,5.742626,47.152498,AA,2339,...,0,200.0,0,INTERNET,3.743048,other,0,1,0,0


,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,...,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.7,0.062288,-1,24,50,0.016740,-0.871747,AB,3430,...,1,200.0,0,INTERNET,6.804428,other,0,1,0,0
1,0,0.9,0.098433,-1,310,50,0.019002,-1.023805,AB,3492,...,1,1500.0,0,INTERNET,1.412211,macintosh,0,1,0,0
2,0,0.6,0.116962,-1,189,60,0.047064,-1.206121,AB,4621,...,0,200.0,0,INTERNET,14.488562,other,1,1,0,0
3,0,0.3,0.059078,10,40,60,0.008007,-0.075908,AA,1697,...,1,200.0,0,INTERNET,6.152497,linux,1,1,0,0
4,0,0.1,0.689959,-1,128,30,2.513544,-1.108190,AD,1431,...,0,200.0,0,INTERNET,5.599853,other,0,1,0,0


**C. Initial Dataset Overview**

The Base and Variant II files both contain 1,000,000 records and 32 columns. Both datasets have the same feature structure. Each dataset contains 11,029 fraud cases, resulting in a fraud rate of approximately 1.10%. This confirms that the problem is highly imbalanced and supports the decision to use PR-AUC / Average Precision as the primary metric instead of accuracy.


The preliminary inspection also shows that the uploaded Base and Variant II files contain no missing cells and no duplicate rows. However, some variables contain values such as -1, which may represent unavailable or not-applicable information rather than true numeric values. These values should be investigated further during preprocessing.

In [ ]:
#Dataset overview and comparison

datasets = {
    "Base": base_df,
    "Variant II": variant_ii_df
}

summary_rows = []

for name, data in datasets.items():
    fraud_count = int(data[TARGET_COL].sum())
    fraud_rate = data[TARGET_COL].mean()

    summary_rows.append({
        "dataset": name,
        "rows": data.shape[0],
        "columns": data.shape[1],
        "fraud_count": fraud_count,
        "fraud_rate_percent": round(fraud_rate * 100, 3),
        "legitimate_count": int((data[TARGET_COL] == 0).sum()),
        "missing_cells": int(data.isna().sum().sum()),
        "duplicate_rows": int(data.duplicated().sum()),
        "numeric_columns": len(data.select_dtypes(include=np.number).columns),
        "categorical_columns": len(data.select_dtypes(exclude=np.number).columns),
        "month_min": int(data["month"].min()),
        "month_max": int(data["month"].max()),
        "unique_months": int(data["month"].nunique())
    })

dataset_overview = pd.DataFrame(summary_rows)
display(dataset_overview)

,dataset,rows,columns,fraud_count,fraud_rate_percent,legitimate_count,missing_cells,duplicate_rows,numeric_columns,categorical_columns,month_min,month_max,unique_months
0,Base,1000000,32,11029,1.103,988971,0,0,27,5,0,7,8
1,Variant II,1000000,32,11029,1.103,988971,0,0,27,5,0,7,8


**d. Column Consistency Check**

Before using Base.csv as the primary dataset and Variant II.csv as the secondary robustness dataset, it is important to confirm that both datasets contain the same columns. This allows the same preprocessing and modeling pipeline to be applied consistently.

In [ ]:
# Check if Base and Variant II have the same columns

base_columns = set(base_df.columns)
variant_ii_columns = set(variant_ii_df.columns)

print("Columns in Base but not in Variant II:")
print(base_columns - variant_ii_columns)

print("\nColumns in Variant II but not in Base:")
print(variant_ii_columns - base_columns)

print("\nSame columns:", base_columns == variant_ii_columns)

Columns in Base but not in Variant II:
set()

Columns in Variant II but not in Base:
set()

Same columns: True


**e. Feature Types**


The dataset contains a combination of numerical, categorical, binary, and time-related variables. The target variable is fraud_bool. The categorical variables include payment_type, employment_status, housing_status, source, and device_os. The remaining variables are numerical or binary.


Understanding feature types is important because each type requires different preprocessing. Categorical variables will require encoding. Numerical variables may require scaling depending on the model. Binary variables should be checked to ensure they contain only expected values. The month variable should be treated as a time-ordering variable because the BAF dataset includes temporal dynamics.

In [ ]:
# Feature type summary using Base dataset

numeric_cols = base_df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = base_df.select_dtypes(exclude=np.number).columns.tolist()

binary_cols = [
    col for col in numeric_cols
    if base_df[col].nunique() == 2 and col != TARGET_COL
]

time_cols = ["month"]

feature_type_summary = pd.DataFrame({
    "feature_group": [
        "Target",
        "Numerical Features",
        "Categorical Features",
        "Binary Features",
        "Time Feature"
    ],
    "columns": [
        [TARGET_COL],
        [col for col in numeric_cols if col not in binary_cols + [TARGET_COL] + time_cols],
        categorical_cols,
        binary_cols,
        time_cols
    ],
    "count": [
        1,
        len([col for col in numeric_cols if col not in binary_cols + [TARGET_COL] + time_cols]),
        len(categorical_cols),
        len(binary_cols),
        len(time_cols)
    ]
})

display(feature_type_summary)

,feature_group,columns,count
0,Target,[fraud_bool],1
1,Numerical Features,"[income, name_email_similarity, prev_address_m...",19
2,Categorical Features,"[payment_type, employment_status, housing_stat...",5
3,Binary Features,"[email_is_free, phone_home_valid, phone_mobile...",6
4,Time Feature,[month],1


**f. Missing Values and Duplicate Records**

The initial inspection of Base.csv and Variant II.csv shows no missing cells and no duplicate rows. However, this does not mean that all values are directly usable. Several numerical features may use -1 as a special value for missing, unavailable, or not-applicable information. These values will be reviewed during preprocessing.

In [ ]:
# Missing values and duplicate checks

quality_rows = []

for name, data in datasets.items():
    quality_rows.append({
        "dataset": name,
        "missing_cells": int(data.isna().sum().sum()),
        "columns_with_missing": int((data.isna().sum() > 0).sum()),
        "duplicate_rows": int(data.duplicated().sum())
    })

data_quality_summary = pd.DataFrame(quality_rows)
display(data_quality_summary)

# Missing value summary by column for Base
missing_summary_base = pd.DataFrame({
    "column": base_df.columns,
    "missing_count": base_df.isna().sum().values,
    "missing_percent": (base_df.isna().mean().values * 100).round(4)
}).sort_values("missing_percent", ascending=False)

display(missing_summary_base)

,dataset,missing_cells,columns_with_missing,duplicate_rows
0,Base,0,0,0
1,Variant II,0,0,0


,column,missing_count,missing_percent
0,fraud_bool,0,0.0
1,income,0,0.0
2,name_email_similarity,0,0.0
3,prev_address_months_count,0,0.0
4,current_address_months_count,0,0.0
5,customer_age,0,0.0
6,days_since_request,0,0.0
7,intended_balcon_amount,0,0.0
8,payment_type,0,0.0
9,zip_count_4w,0,0.0


**g. Target Distribution and Class Imbalance**

The target variable is fraud_bool. In both uploaded datasets, the fraud rate is approximately 1.10%, which means the dataset is highly imbalanced. This confirms the metric decision made: accuracy will be reported but will not be used as the primary metric. PR-AUC / Average Precision, recall, precision, F1-score, and recall at top-K% are more appropriate for evaluating fraud detection performance.

In [ ]:
# Target distribution

target_rows = []

for name, data in datasets.items():
    counts = data[TARGET_COL].value_counts().sort_index()
    percents = data[TARGET_COL].value_counts(normalize=True).sort_index() * 100

    for class_value in counts.index:
        target_rows.append({
            "dataset": name,
            "class": class_value,
            "class_label": "fraud" if class_value == 1 else "legitimate",
            "count": int(counts[class_value]),
            "percent": round(percents[class_value], 3)
        })

target_distribution = pd.DataFrame(target_rows)
display(target_distribution)

,dataset,class,class_label,count,percent
0,Base,0,legitimate,988971,98.897
1,Base,1,fraud,11029,1.103
2,Variant II,0,legitimate,988971,98.897
3,Variant II,1,fraud,11029,1.103


**h. Categorical Value Review**

The categorical variables in the dataset are encoded using category labels. These variables will require encoding before model training. The categorical variables are payment_type, employment_status, housing_status, source, and device_os.

In [ ]:
# Categorical value review

categorical_summary_rows = []

for name, data in datasets.items():
    for col in categorical_cols:
        categorical_summary_rows.append({
            "dataset": name,
            "column": col,
            "unique_values_count": data[col].nunique(),
            "unique_values": sorted(data[col].dropna().unique().tolist())
        })

categorical_summary = pd.DataFrame(categorical_summary_rows)
display(categorical_summary)

,dataset,column,unique_values_count,unique_values
0,Base,payment_type,5,"[AA, AB, AC, AD, AE]"
1,Base,employment_status,7,"[CA, CB, CC, CD, CE, CF, CG]"
2,Base,housing_status,7,"[BA, BB, BC, BD, BE, BF, BG]"
3,Base,source,2,"[INTERNET, TELEAPP]"
4,Base,device_os,5,"[linux, macintosh, other, windows, x11]"
5,Variant II,payment_type,5,"[AA, AB, AC, AD, AE]"
6,Variant II,employment_status,7,"[CA, CB, CC, CD, CE, CF, CG]"
7,Variant II,housing_status,7,"[BA, BB, BC, BD, BE, BF, BG]"
8,Variant II,source,2,"[INTERNET, TELEAPP]"
9,Variant II,device_os,5,"[linux, macintosh, other, windows, x11]"


**i. Numerical Range, Sentinel Values, and Initial Outlier Check**


The dataset contains numerical variables with different scales and meanings. Some variables, such as velocity indicators and session length, may have wide ranges or skewed distributions. Other variables, such as address month counts, bank month counts, device distinct emails, and session length, may contain -1, which should be reviewed as a possible sentinel value.


For this capstone, the goal is not yet to treat outliers, but to identify which variables may need further attention during preprocessing and feature engineering.

In [ ]:
# Numerical range and sentinel value check

numeric_summary_rows = []

for col in numeric_cols:
    numeric_summary_rows.append({
        "column": col,
        "dtype": str(base_df[col].dtype),
        "base_min": base_df[col].min(),
        "base_max": base_df[col].max(),
        "variant_ii_min": variant_ii_df[col].min(),
        "variant_ii_max": variant_ii_df[col].max(),
        "base_unique_values": base_df[col].nunique(),
        "variant_ii_unique_values": variant_ii_df[col].nunique(),
        "base_negative_one_count": int((base_df[col] == -1).sum()),
        "variant_ii_negative_one_count": int((variant_ii_df[col] == -1).sum())
    })

numeric_range_summary = pd.DataFrame(numeric_summary_rows)
display(numeric_range_summary)

,column,dtype,base_min,base_max,variant_ii_min,variant_ii_max,base_unique_values,variant_ii_unique_values,base_negative_one_count,variant_ii_negative_one_count
0,fraud_bool,int64,0.000000e+00,1.000000,0.000000e+00,1.000000,2,2,0,0
1,income,float64,1.000000e-01,0.900000,1.000000e-01,0.900000,9,9,0,0
2,name_email_similarity,float64,1.434550e-06,0.999999,5.024707e-08,1.000000,998861,998708,0,0
3,prev_address_months_count,int64,-1.000000e+00,383.000000,-1.000000e+00,399.000000,374,372,712920,759415
4,current_address_months_count,int64,-1.000000e+00,428.000000,-1.000000e+00,429.000000,423,418,4254,3488
5,customer_age,int64,1.000000e+01,90.000000,1.000000e+01,90.000000,9,9,0,0
6,days_since_request,float64,4.036860e-09,78.456904,3.112791e-08,76.577505,989330,989023,0,0
7,intended_balcon_amount,float64,-1.553055e+01,112.956928,-1.573989e+01,112.702504,994971,994785,0,0
8,zip_count_4w,int64,1.000000e+00,6700.000000,1.000000e+00,6650.000000,6306,6315,0,0
9,velocity_6h,float64,-1.706031e+02,16715.565404,-1.741097e+02,16801.339834,998687,998677,0,0


In [ ]:
# Initial IQR-based outlier check for Base dataset

outlier_rows = []

for col in numeric_cols:
    if col == TARGET_COL:
        continue

    q1 = base_df[col].quantile(0.25)
    q3 = base_df[col].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = int(((base_df[col] < lower_bound) | (base_df[col] > upper_bound)).sum())
    outlier_percent = outlier_count / len(base_df) * 100

    outlier_rows.append({
        "column": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": outlier_count,
        "outlier_percent": round(outlier_percent, 3)
    })

outlier_summary_base = pd.DataFrame(outlier_rows).sort_values("outlier_percent", ascending=False)
display(outlier_summary_base)

,column,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_percent
19,proposed_credit_limit,200.000000,500.000000,300.000000,-250.000000,950.000000,241742,24.174
18,has_other_cards,0.000000,0.000000,0.000000,0.000000,0.000000,222988,22.299
6,intended_balcon_amount,-1.181488,4.984176,6.165664,-10.429984,14.232671,222702,22.270
11,bank_branch_count_8w,1.000000,25.000000,24.000000,-35.000000,61.000000,175243,17.524
2,prev_address_months_count,-1.000000,12.000000,13.000000,-20.500000,31.500000,157320,15.732
16,phone_mobile_valid,1.000000,1.000000,0.000000,1.000000,1.000000,110324,11.032
5,days_since_request,0.007193,0.026331,0.019137,-0.021513,0.055037,94834,9.483
21,session_length_in_minutes,3.103053,8.866131,5.763078,-5.541564,17.510748,78789,7.879
7,zip_count_4w,894.000000,1944.000000,1050.000000,-681.000000,3519.000000,59871,5.987
3,current_address_months_count,19.000000,130.000000,111.000000,-147.500000,296.500000,41001,4.100


**j. Data Dictionary**


The data dictionary below documents the variables, expected data type, description, allowed values or range, and planned use. This will guide preprocessing and feature engineering.

In [ ]:
# Data dictionary

data_dictionary = pd.DataFrame([
    {
        "variable": "fraud_bool",
        "type": "Binary target",
        "description": "Indicates whether the application is fraudulent.",
        "allowed_values_or_range": "0 = legitimate, 1 = fraud",
        "planned_use": "Target variable"
    },
    {
        "variable": "income",
        "type": "Numerical / ordinal",
        "description": "Applicant income value represented in scaled or binned form.",
        "allowed_values_or_range": "0.1 to 0.9",
        "planned_use": "Predictor; may also be used for fairness/proxy analysis"
    },
    {
        "variable": "name_email_similarity",
        "type": "Numerical",
        "description": "Similarity between applicant name and email address.",
        "allowed_values_or_range": "0 to 1",
        "planned_use": "Identity consistency predictor"
    },
    {
        "variable": "prev_address_months_count",
        "type": "Numerical",
        "description": "Number of months associated with the previous address.",
        "allowed_values_or_range": "Integer; -1 may indicate unavailable/not applicable",
        "planned_use": "Address history predictor; check sentinel values"
    },
    {
        "variable": "current_address_months_count",
        "type": "Numerical",
        "description": "Number of months associated with the current address.",
        "allowed_values_or_range": "Integer; -1 may indicate unavailable/not applicable",
        "planned_use": "Address stability predictor"
    },
    {
        "variable": "customer_age",
        "type": "Numerical / ordinal",
        "description": "Applicant age or age group.",
        "allowed_values_or_range": "Values such as 10 to 90",
        "planned_use": "Predictor and fairness analysis variable"
    },
    {
        "variable": "days_since_request",
        "type": "Numerical",
        "description": "Elapsed time since the application/request event.",
        "allowed_values_or_range": "Continuous numeric",
        "planned_use": "Application timing predictor"
    },
    {
        "variable": "intended_balcon_amount",
        "type": "Numerical",
        "description": "Intended balance or account amount indicator.",
        "allowed_values_or_range": "Continuous numeric; may include negative values",
        "planned_use": "Financial behavior predictor; check skew/outliers"
    },
    {
        "variable": "payment_type",
        "type": "Categorical",
        "description": "Payment or application payment category.",
        "allowed_values_or_range": "AA, AB, AC, AD, AE",
        "planned_use": "Categorical predictor; encode before modeling"
    },
    {
        "variable": "zip_count_4w",
        "type": "Numerical",
        "description": "Count of activity associated with zip code over four weeks.",
        "allowed_values_or_range": "Integer",
        "planned_use": "Velocity or geographic activity predictor"
    },
    {
        "variable": "velocity_6h",
        "type": "Numerical",
        "description": "Activity velocity over six hours.",
        "allowed_values_or_range": "Continuous numeric",
        "planned_use": "Short-term behavioral velocity predictor"
    },
    {
        "variable": "velocity_24h",
        "type": "Numerical",
        "description": "Activity velocity over twenty-four hours.",
        "allowed_values_or_range": "Continuous numeric",
        "planned_use": "Daily behavioral velocity predictor"
    },
    {
        "variable": "velocity_4w",
        "type": "Numerical",
        "description": "Activity velocity over four weeks.",
        "allowed_values_or_range": "Continuous numeric",
        "planned_use": "Medium-term behavioral velocity predictor"
    },
    {
        "variable": "bank_branch_count_8w",
        "type": "Numerical",
        "description": "Count of bank branch-related activity over eight weeks.",
        "allowed_values_or_range": "Integer",
        "planned_use": "Behavioral/activity predictor"
    },
    {
        "variable": "date_of_birth_distinct_emails_4w",
        "type": "Numerical",
        "description": "Number of distinct emails associated with the same date of birth over four weeks.",
        "allowed_values_or_range": "Integer",
        "planned_use": "Identity risk predictor"
    },
    {
        "variable": "employment_status",
        "type": "Categorical",
        "description": "Applicant employment status category.",
        "allowed_values_or_range": "CA, CB, CC, CD, CE, CF, CG",
        "planned_use": "Categorical predictor; possible fairness/proxy analysis"
    },
    {
        "variable": "credit_risk_score",
        "type": "Numerical",
        "description": "Credit risk score or creditworthiness indicator.",
        "allowed_values_or_range": "Integer; may include negative values",
        "planned_use": "Credit-risk predictor"
    },
    {
        "variable": "email_is_free",
        "type": "Binary",
        "description": "Indicates whether the applicant email uses a free/public email domain.",
        "allowed_values_or_range": "0 = no, 1 = yes",
        "planned_use": "Identity/contact predictor"
    },
    {
        "variable": "housing_status",
        "type": "Categorical",
        "description": "Applicant housing status category.",
        "allowed_values_or_range": "BA, BB, BC, BD, BE, BF, BG",
        "planned_use": "Categorical predictor; possible fairness/proxy analysis"
    },
    {
        "variable": "phone_home_valid",
        "type": "Binary",
        "description": "Indicates whether the home phone is valid.",
        "allowed_values_or_range": "0 = invalid, 1 = valid",
        "planned_use": "Contact validity predictor"
    },
    {
        "variable": "phone_mobile_valid",
        "type": "Binary",
        "description": "Indicates whether the mobile phone is valid.",
        "allowed_values_or_range": "0 = invalid, 1 = valid",
        "planned_use": "Contact validity predictor"
    },
    {
        "variable": "bank_months_count",
        "type": "Numerical",
        "description": "Number of months associated with bank history.",
        "allowed_values_or_range": "Integer; -1 may indicate unavailable/not applicable",
        "planned_use": "Banking history predictor; check sentinel values"
    },
    {
        "variable": "has_other_cards",
        "type": "Binary",
        "description": "Indicates whether the applicant has other cards.",
        "allowed_values_or_range": "0 = no, 1 = yes",
        "planned_use": "Financial profile predictor"
    },
    {
        "variable": "proposed_credit_limit",
        "type": "Numerical",
        "description": "Proposed credit limit for the application.",
        "allowed_values_or_range": "Numeric values such as 190 to 2100",
        "planned_use": "Credit exposure predictor"
    },
    {
        "variable": "foreign_request",
        "type": "Binary",
        "description": "Indicates whether the request came from a foreign source/location.",
        "allowed_values_or_range": "0 = no, 1 = yes",
        "planned_use": "Request origin predictor"
    },
    {
        "variable": "source",
        "type": "Categorical",
        "description": "Application source or channel.",
        "allowed_values_or_range": "INTERNET, TELEAPP",
        "planned_use": "Channel predictor; encode before modeling"
    },
    {
        "variable": "session_length_in_minutes",
        "type": "Numerical",
        "description": "Length of the application session in minutes.",
        "allowed_values_or_range": "Continuous numeric; -1 may indicate unavailable/not applicable",
        "planned_use": "Session behavior predictor"
    },
    {
        "variable": "device_os",
        "type": "Categorical",
        "description": "Operating system of the device used during application.",
        "allowed_values_or_range": "linux, macintosh, other, windows, x11",
        "planned_use": "Device predictor; encode before modeling"
    },
    {
        "variable": "keep_alive_session",
        "type": "Binary",
        "description": "Indicates whether the session was kept alive.",
        "allowed_values_or_range": "0 = no, 1 = yes",
        "planned_use": "Session behavior predictor"
    },
    {
        "variable": "device_distinct_emails_8w",
        "type": "Numerical",
        "description": "Number of distinct emails associated with the same device over eight weeks.",
        "allowed_values_or_range": "-1, 0, 1, 2",
        "planned_use": "Device identity risk predictor; check sentinel values"
    },
    {
        "variable": "device_fraud_count",
        "type": "Numerical",
        "description": "Prior fraud count associated with the device.",
        "allowed_values_or_range": "Integer; uploaded files show this may be constant",
        "planned_use": "Assess for low variance; remove later if constant"
    },
    {
        "variable": "month",
        "type": "Time / ordinal",
        "description": "Month index used for temporal ordering.",
        "allowed_values_or_range": "0 to 7",
        "planned_use": "Time-based validation, drift review, and temporal split"
    }
])

display(data_dictionary)

,variable,type,description,allowed_values_or_range,planned_use
0,fraud_bool,Binary target,Indicates whether the application is fraudulent.,"0 = legitimate, 1 = fraud",Target variable
1,income,Numerical / ordinal,Applicant income value represented in scaled o...,0.1 to 0.9,Predictor; may also be used for fairness/proxy...
2,name_email_similarity,Numerical,Similarity between applicant name and email ad...,0 to 1,Identity consistency predictor
3,prev_address_months_count,Numerical,Number of months associated with the previous ...,Integer; -1 may indicate unavailable/not appli...,Address history predictor; check sentinel values
4,current_address_months_count,Numerical,Number of months associated with the current a...,Integer; -1 may indicate unavailable/not appli...,Address stability predictor
5,customer_age,Numerical / ordinal,Applicant age or age group.,Values such as 10 to 90,Predictor and fairness analysis variable
6,days_since_request,Numerical,Elapsed time since the application/request event.,Continuous numeric,Application timing predictor
7,intended_balcon_amount,Numerical,Intended balance or account amount indicator.,Continuous numeric; may include negative values,Financial behavior predictor; check skew/outliers
8,payment_type,Categorical,Payment or application payment category.,"AA, AB, AC, AD, AE",Categorical predictor; encode before modeling
9,zip_count_4w,Numerical,Count of activity associated with zip code ove...,Integer,Velocity or geographic activity predictor


**k. Initial Data Understanding and Findings**


The initial data understanding shows that both Base.csv and Variant II.csv contain the same 32 columns and 1,000,000 records each. Both datasets have the same number of fraud cases and a fraud rate of approximately 1.10%, confirming that this is a highly imbalanced fraud detection problem. This supports the earlier decision to use PR-AUC / Average Precision as the primary metric.


The dataset includes numerical, categorical, binary, and time-related variables. Categorical features will require encoding in the next step. Numerical features have different scales and may require scaling for models such as Logistic Regression. Tree-based models such as Random Forest and XGBoost may not require the same level of scaling, but consistent preprocessing will still be documented.


The dataset contains no missing cells and no duplicate rows in the uploaded files. However, some variables contain -1, which should be treated carefully as a potential sentinel value. These variables will be reviewed during preprocessing. The dataset also includes a month variable from 0 to 7, which enables time-based validation. A time-based split will be considered in later modeling to better simulate real-world fraud detection, where models are trained on historical applications and used on future applications.

**Summary**

Base and Variant II each contain 1,000,000 applications, 32 original columns, and 11,029 fraud cases, corresponding to a fraud rate of 1.103%. The schemas are identical, with 27 numeric and five categorical variables covering application, identity, address, credit, behavior, device, channel, and temporal information. Neither file contains conventional null values or duplicate rows.


The absence of blank nulls did not mean the data was complete. Several fields use -1 as a sentinel value, most notably prev_address_months_count in 71.29% of Base and 75.94% of Variant II, and bank_months_count in approximately one quarter of records. These values required explicit missingness indicators and controlled imputation. IQR review also found many statistical outliers, but they were not automatically deleted because unusual activity may carry genuine fraud information.


The target distribution confirms severe imbalance: 98.897% legitimate and 1.103% fraudulent. This finding justifies PR-AUC, top-K fraud capture, class weighting, and a capacity-based threshold rather than accuracy as the main success criterion. The data dictionary and categorical-value review establish a reproducible basis for preprocessing and feature engineering.